# ![icon](./images/uva-icon-57x57.png) WEEK 11 DBT (Data Build Tool)

| **Working Efficiently With Software** |
| :--: |
| **Data Engineering** |
| **School of Data Science** |
| **University of Virginia** |

# ![icon](./images/uva-icon-57x57.png) Announcements & Agenda

---

## **STILL Looking for volunteers to talk about the use of Claude in the class**
**We need at least 2 students.  3 better, but 2 is good**


---

## **Question about docker lab**
**Check your docker file and makefile into a branch just like previou labs.  Put your link to the dockerhub image in the PR description**

---


## T-Shirt sizing the remaining labs.
- ~~Setting up snowflake connector.  **M/L**~~
- ~~Dockerizing our pipeline.~~  **XL**
- ~~Git repo in snowflake.~~  **M**
- DBT-expectations **M**   **<===== AWS ⬆️ == Snowflake ⬇️ ===**

## We'll probably do this lab "in class" on the 4th.
- Snowflake CORTEX.  **S**

# Step-by-Step Live DBT Walkthrough

# ![icon](./images/uva-icon-57x57.png) 1. Environment Setup & Installation

Run in your local terminal or EC2 instance:

```bash
pip install dbt-snowflake

```

* **NB:** `dbt-core` is the open-source engine, while `dbt-snowflake` is the specific database adapter. dbt uses adapter packages to translate generic dbt commands into Snowflake-native SQL dialects.

# ![icon](./images/uva-icon-57x57.png) 2. Initializing Configuration (`dbt init`)

## Run:

```bash
dbt init demo

```

* **Prompts to Walk Through Live:**
* *Which database?* Select `snowflake`.
* *Account:* Enter your Snowflake account locator (e.g., `rja95216`).
* *User / Password:* Enter student/demo credentials.
* *Role:* `ds5111_student_role`.
* *Warehouse:* `DS5111_WH`.
* *Database / Schema:* `DS5111_DB` / `TXT1SR`.
* *Authenticator:* Select `username_password_mfa` (or standard credentials).



Show them the generated `~/.dbt/profiles.yml` file:

```yaml
demo:
  target: dev
  outputs:
    dev:
      type: snowflake
      account: rja95216 
      user: TXT1SR
      password: *****
      role: ds5111_student_role
      database: DS5111_DB
      schema: TXT1SR
      warehouse: DS5111_WH
      authenticator: username_password_mfa
      threads: 1

```

> **NB:** *"Notice `authenticator: username_password_mfa`. When we fire off a command, watch your phone—you must approve the **Duo Push notification** immediately, or the CLI connection times out!"*

# ![icon](./images/uva-icon-57x57.png) 3. Verify Connection (`dbt debug`)

## **NB:** Every time you do anything that connects, you'll have to use your authenticator!

Run:

```bash
dbt debug

```

* **NB:** `dbt debug` verifies four critical dependencies:
1. Python environment.
2. `dbt_project.yml` structure.
3. `profiles.yml` syntax.
4. Database network connectivity and authentication.

# ![icon](./images/uva-icon-57x57.png) 4. Database Inspection & The First Build

1. Open Snowflake UI live and show the `TXT1SR` schema: **No tables or views created by dbt exist yet.**
2. Run the foundational build command:
```bash
dbt build

```



* **NB:** `dbt build` is a single command that runs models (`dbt run`), executes tests (`dbt test`), and populates seeds (`dbt seed`) in DAG dependency order.

# ![icon](./images/uva-icon-57x57.png) 5. Inspecting Built Models & Handling Test Failures

1. CLI output will show you a built-in default test fails (or passes).
2. Open Snowflake UI: You now have newly created `my_first_dbt_model` and `my_second_dbt_model` views/tables.
3. Inspect `models/example/schema.yml` in your editor. Note how tests like `not_null` and `unique` are declared in YAML.
4. Modify the model or data to resolve the test failure:
```sql
/*
    Welcome to your first dbt model!
    Did you know that you can also configure models directly within SQL files?
    This will override configurations stated in dbt_project.yml

    Try changing "table" to "view" below
*/

{{ config(materialized='table') }}

with source_data as (

    select 1 as id
    union all
    select null as id   -- NB: This is what causes the failure

)

select *
from source_data

/*
    Uncomment the line below to remove records with null `id` values
*/

-- where id is not null
```

then re-run:
```bash
dbt build

```
5. You should now see the green passing status in the terminal.

# ![icon](./images/uva-icon-57x57.png)  6. Complex Jinja Demo: Dynamic Pivot Model

## Drop the complex Jinja model into `models/mart_tech_term_pivot.sql`:

```sql
{{ config(materialized='table') }}

{% set core_terms = ['python', 'sql', 'dbt', 'snowflake', 'aws', 'docker'] %}

SELECT
    video_id,
    {% for term in core_terms %}
    SUM(CASE WHEN LOWER(tech_term) = '{{ term }}' THEN 1 ELSE 0 END) AS count_{{ term }}_mentions
    {% if not loop.last %},{% endif %}
    {% endfor %}
-- FROM {{ ref('fct_tech_terms') }}  NB:!!!! This notation tells DBT it's a defined model
FROM {{ source('my_database_sources', 'FCT_TECH_TERMS') }}  -- This notation says it pre-exists
GROUP BY video_id

```

## Pre existing tables go in sources.yml
```yaml
version: 2

sources:
  - name: my_database_sources # You can name this whatever makes sense
    schema: TXT1SR # The schema shown in your screenshot
    tables:
      - name: FCT_TECH_TERMS
```




Run the model:

```bash
dbt run --select mart_tech_term_pivot

```

## The table should now show up in snowflake

## Let's inspect the 'compiled' version
**Look in:** `target/compiled/demo_project/models/mart_tech_term_pivot.sql`

# ![icon](./images/uva-icon-57x57.png) 7. Adding a Test to the Dynamic Jinja Model

Show how effortless it is to add data testing to the new Jinja model by appending it to `models/schema.yml`:

```yaml
models:
  - name: mart_tech_term_pivot
    columns:
      - name: video_id
        tests:
          - not_null
          - unique

```

Run the test suite specifically for this model:

```bash
dbt test --select mart_tech_term_pivot

```

Point out how dbt dynamically generates a SQL query checking for `COUNT(*) > 1` or `IS NULL` behind the scenes.

# ![icon](./images/uva-icon-57x57.png)  Technical Deep Dives & Lecture Explanations

### Deconstructing `dbt_project.yml` the "brain" of the project directory.

```yaml
name: 'ds5111_transforms'    # Project identifier (used in ref() functions)
version: '1.0.0'             # Semantic versioning of your data project
config-version: 2           # Specifies dbt project syntax version
profile: 'default'           # Connects this project to a specific block in profiles.yml

model-paths: ["models"]      # Where dbt looks for SQL models
test-paths: ["tests"]        # Where custom data test queries live
macro-paths: ["macros"]      # Where reusable Jinja macros live

models:
  ds5111_transforms:
    # Applies materialization defaults to subfolders inside models/
    staging:
      +materialized: view    # Staging models default to lightweight views
    marts:
      +materialized: table   # Data marts default to physical tables

```

# ![icon](./images/uva-icon-57x57.png)  Core Conceptual Takeaways


#### 1. Platform-Agnostic Transformations

* The models and Jinja code written in dbt are **decoupled from the underlying database platform**.
* The exact same dbt model code can run on [**Snowflake, BigQuery, Databricks, PostgreSQL, or Amazon Redshift**](https://docs.getdbt.com/docs/trusted-adapters?version=2.0) simply by changing the `type:` entry in `profiles.yml`. dbt translates generic functions into target-specific SQL dialects.

#### 2. Infrastructure as Code & Git-Native Analytics

* Data transformations are no longer hidden inside GUI tools or manually executed ad-hoc SQL scripts.
* Everything—table schemas, transformations, documentation, and data quality tests—lives as **text files inside a Git repository**.
* This enables software engineering best practices: version control, code reviews (PRs), CI/CD pipelines, and automated testing before merging code to production.

#### 3. Why We Use Snowflake-Native Execution in Lab 11

Note the differences between the CLI and the in Snowflake approach:

* **No local environment drift:** Avoids local Python dependency issues, driver versions, and virtual environment setup.
* **No server management:** Students don't need to spin up and maintain EC2 instances or SSH keys.
* **No MFA friction:** Avoids repeated Duo Push prompts on every single `dbt run` CLI command. Snowflake native execution inherits your authenticated web session automatically.
* **GitOps Workflow:** Pushing code to GitHub and running `ALTER GIT REPOSITORY ... FETCH` directly inside Snowflake gives you the exact same Git-native guarantees without local environment overhead.

# 🏗️ The dbt Ecosystem: Deployment Strategies

> **Conceptual Checkpoint:** We know *what* dbt does (transforms data in the warehouse), but *how* do data teams actually run it in production? 

While we are currently using snowflake and you've now seen the terminal to execute commands, engineering teams generally choose from three primary deployment paths based on their budget, infrastructure, and technical comfort levels.

| Feature | 💻 dbt Core (CLI) | ☁️ dbt Cloud | ❄️ Native Snowflake |
| :--- | :--- | :--- | :--- |
| **The Architecture** | The open-source Python package running on local machines or external servers. | A fully managed SaaS platform hosted by dbt Labs. | Running dbt Core inside Snowpark Container Services (SPCS). |
| **The Interface** | Terminal, Vim, VS Code. | Browser-based Web IDE. | Terminal, Snowflake UI. |
| **Job Orchestration** | Bring your own (cron, Airflow, Dagster, GitHub Actions). | Built-in native scheduler and automated CI/CD pipelines. | Scheduled via Snowflake Tasks. |
| **Cost** | Free (Open Source). | Per-seat subscription pricing. | Snowflake compute credits. |
| **The Target Audience** | Engineers who want complete control over their infrastructure and CI/CD pipelines. | Analytics teams that want an out-of-the-box solution without managing servers. | Teams trying to keep 100% of their compute and data movement strictly inside Snowflake. |

---

## Why you should explore the CLI?
Using the CLI on an EC2 instance because it forces you to understand the deeper underlying mechanics of dbt. By learning how the file system maps to the data warehouse, how YAML configurations tie models together, and how to debug compile errors in a headless environment, you build a much stronger mental model than if a web interface was hiding the hard parts.  The Snowflake approach we use in the lab focuses on just the right aspects you need to work with it.  However, the flexibility of using the command line is not paralleled, you can switch vendors in a snap.  **Plus, as you probably guessed already, you can automate and combind this with github actions.**